# Frozen Lake Dynamic Programming Experiments

This notebook runs policy iteration and value iteration on Gymnasium `FrozenLake-v1` with `is_slippery=True`. The environment is always the Gymnasium environment; the reward variants are supplied through Gymnasium's `reward_schedule` argument instead of a custom environment.

## Setup

In [ ]:
import os
import sys
from pathlib import Path

for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "src" / "rl_suite").is_dir():
        sys.path.insert(0, str(_p / "src"))
        break

os.environ.setdefault("MPLCONFIGDIR", "/private/tmp/matplotlib")
os.environ.setdefault("XDG_CACHE_HOME", "/private/tmp")

from IPython.display import Markdown, display

from rl_suite.experiments.frozen_lake import (
    ACTIONS,
    REWARDS,
    make_env,
    plot_summary,
    plot_values,
    named_policy,
    run_experiments,
)

%matplotlib inline

In [ ]:
SEED = 10
GAMMAS = (0.9, 0.99)
REWARD_SCHEMES = ("sparse", "dense")
EVAL_EPISODES = 100

## Environment

The transition model below is Gymnasium's slippery 4x4 Frozen Lake. Each intended move can slip to a lateral direction, so dynamic programming must sum over every `(probability, next_state, reward, terminated)` tuple in `env.unwrapped.P[s][a]`.

In [ ]:
env = make_env("sparse")
print("Reward schedules:", REWARDS)
print("Actions:", dict(enumerate(ACTIONS)))
print("Map:")
print("\n".join(b"".join(row).decode("utf-8") for row in env.unwrapped.desc))
print("\nTransitions from state 0:")
env.unwrapped.P[0]

## Run Experiments

In [ ]:
experiments = run_experiments(gammas=GAMMAS, rewards=REWARD_SCHEMES, episodes=EVAL_EPISODES)

In [ ]:
headers = ["algorithm", "reward", "gamma", "theta", "iterations", "sweeps", "runtime_ms", "success_rate"]
lines = ["| " + " | ".join(headers) + " |", "| " + " | ".join(["---"] * len(headers)) + " |"]

for exp in experiments:
    row = [
        exp["algorithm"], exp["reward"], f"{exp['gamma']:.2f}", f"{exp['theta']:.0e}",
        exp["iterations"], exp["sweeps"], f"{exp['runtime'] * 1000:.3f}", f"{exp['success_rate']:.2%}",
    ]
    lines.append("| " + " | ".join(map(str, row)) + " |")

display(Markdown("\n".join(lines)))

## Best Runtime Configuration

The cell below selects the fastest measured configuration from this run and evaluates its greedy policy again with the same stochastic Gymnasium dynamics.

In [ ]:
best = min(experiments, key=lambda exp: exp["runtime"])

print(f"Best runtime: {best['algorithm']} with reward={best['reward']}, gamma={best['gamma']}")
print(f"Success rate over {EVAL_EPISODES} slippery episodes: {best['success_rate']:.2%}")
print("Latest trajectory:", best["latest_path"])
print("Policy grid:")
named_policy(best["policy"])

## Plots

In [ ]:
plot_summary(experiments)

In [ ]:
plot_values(experiments)

## Notes

- Higher `gamma` values usually require more evaluation work because future rewards decay more slowly, so value estimates keep changing for longer.
- Dense rewards add immediate feedback for holes and steps. In this small tabular problem that can change the number of sweeps, but the runtime differences are tiny enough that repeated timing can vary.
- Policy iteration stores a value vector and a policy vector, so its algorithm-specific memory is `O(|S|)`. Value iteration stores a value vector and derives the greedy policy at the end, so it is also `O(|S|)`.